# Preparació dataset 2

## Retallar àrea del video

### Càlcul àrea a utilitzar

In [ ]:
import cv2
import sys

def main():
    # Vídeo d'exemple per provar
    video_path = 'room1_cam2_2026-04-19_08-05-10.ts'

    # Obrim el vídeo
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error obrint: {video_path}")
        return

    # Llegim la primera imatge (frame)
    ret, frame = cap.read()
    if not ret:
        print("Error al llegir el vídeo.")
        return

    print("Selecciona la zona amb el ratolí i prem ENTER ('c' per cancel·lar).")

    # Deixem que l'usuari seleccioni l'àrea
    window_name = "Tria la zona a retallar"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    # Ajustem la mida de la finestra perquè es vegi bé a la pantalla
    cv2.resizeWindow(window_name, 1280, 720)

    roi = cv2.selectROI(window_name, frame, showCrosshair=True, fromCenter=False)
    cv2.destroyAllWindows()

    x, y, w, h = roi
    if w == 0 or h == 0:
        print("Cap zona seleccionada. Adéu!")
        return

    print(f"\nResultat -> x:{x}, y:{y}, w:{w}, h:{h}")
    print(f"Codi: frame[{y}:{y+h}, {x}:{x+w}]\n")

    print("Mostrant resultat 10s ('q' per sortir)...")

    # Reproduïm una mica el vídeo retallat per veure com queda
    frames_shown = 0
    while cap.isOpened() and frames_shown < 300: # Uns 10 segons si va a 30fps
        ret, frame = cap.read()
        if not ret:
            break

        # Retallem la imatge actual
        cropped_frame = frame[y:y+h, x:x+w]

        cv2.namedWindow("Vídeo Retallat", cv2.WINDOW_NORMAL)
        # Fem la finestra més petita si l'àrea és massa gran
        window_width = min(w, 1280)
        window_height = min(h, 720)
        cv2.resizeWindow("Vídeo Retallat", window_width, window_height)
        cv2.imshow("Vídeo Retallat", cropped_frame)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

        frames_shown += 1

    cap.release()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    main()

### Retallar video

In [ ]:
import cv2
import os
import glob

def process_videos():
    # Coordenades de retall per defecte (posa aquí les teves)
    x, y = 310, 172
    w, h = 1332, 822

    # Busquem tots els vídeos '.ts' que hi hagi a la carpeta
    video_files = glob.glob('*.ts')

    if not video_files:
        print("No hi ha vídeos .ts")
        return

    print(f"Vídeos trobats: {len(video_files)}")

    # Creem una carpeta pels vídeos nous si no existeix
    os.makedirs("Cropped_Videos", exist_ok=True)

    for file in video_files:
        print(f"\nProcessant: {file}")

        # Obrim el vídeo
        cap = cv2.VideoCapture(file)
        if not cap.isOpened():
            print(f"Error amb: {file}")
            continue

        # Mirem a quants frames per segon va (FPS)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps == 0:
            fps = 30.0 # Posem 30 per defecte si falla

        # Preparem com guardarem el nou vídeo (en format .mp4)
        output_name = os.path.join("Cropped_Videos", file.replace('.ts', '_cropped.mp4'))
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_name, fourcc, fps, (w, h))

        # Comptem els frames totals per mostrar el progrés
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames_processed = 0

        while True:
            # Llegim un frame
            ret, frame = cap.read()
            if not ret:
                break

            # Retallem el frame amb les coordenades
            cropped_frame = frame[y:y+h, x:x+w]

            # Guardem el frame retallat al nou vídeo
            out.write(cropped_frame)

            frames_processed += 1
            if frames_processed % 100 == 0:
                print(f"  Fet: {frames_processed}/{total_frames}", end='\r')

        print(f"\n  Guardat com: {output_name}")

        # Tanquem els fitxers d'aquest vídeo
        cap.release()
        out.release()

    print("\n--- RESUM ---")
    print("Vídeos retallats correctament!")
    print("Estan a la carpeta 'Cropped_Videos'.")
    print("-------------")

if __name__ == '__main__':
    process_videos()

## Extracció de frames

In [ ]:
import cv2
import os
import glob

def extract_n_frames(input_folder, output_folder, target_num_frames=130):
    """
    Agafa N imatges (frames) repartides per igual entre tots els vídeos de la carpeta.
    """
    # Creem la carpeta on guardarem les imatges si cal
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Esborrem imatges velles perquè no es barregin amb les noves
    print("Esborrant fotos velles...")
    for f in glob.glob(os.path.join(output_folder, "*.jpg")):
        try:
            os.remove(f)
        except Exception as e:
            pass

    # Llista de tots els vídeos que tenim
    video_files = glob.glob(os.path.join(input_folder, "*.mp4")) + glob.glob(os.path.join(input_folder, "*.ts"))

    if not video_files:
        print(f"Carpeta buida: {input_folder}")
        return

    # Comptem quants frames tenim en total ajuntant tots els vídeos
    total_frames = 0
    for video_path in video_files:
        cap = cv2.VideoCapture(video_path)
        if cap.isOpened():
            total_frames += int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

    if total_frames == 0:
        print("Error al llegir els vídeos.")
        return

    # Calculem cada quantes imatges n'hem de guardar una
    frames_to_skip = max(1, total_frames // target_num_frames)
    print(f"Total: {total_frames} fotos. En guardem 1 de cada {frames_to_skip}.")

    saved_count = 0
    global_frame_idx = 0

    for video_path in video_files:
        # Si ja tenim les imatges que volíem, parem
        if saved_count >= target_num_frames:
            break

        video_name = os.path.basename(video_path)
        cap = cv2.VideoCapture(video_path)

        while cap.isOpened() and saved_count < target_num_frames:
            # Llegim la imatge
            ret, frame = cap.read()
            if not ret:
                break

            # Si ens toca guardar-la segons el salt que hem calculat, la guardem
            if global_frame_idx % frames_to_skip == 0:
                out_name = f"frame_cam2_{saved_count:03d}.jpg"
                out_path = os.path.join(output_folder, out_name)
                cv2.imwrite(out_path, frame)
                saved_count += 1

            global_frame_idx += 1

        cap.release()

    print(f"Fet! {saved_count} fotos guardades a '{output_folder}'.")

if __name__ == "__main__":
    INPUT_FOLDER = "Cropped_Videos"
    OUTPUT_FOLDER = "frames_cam2_yolov8"
    TARGET_FRAMES = 120

    print(f"Anem a extreure {TARGET_FRAMES} fotos...")
    extract_n_frames(INPUT_FOLDER, OUTPUT_FOLDER, TARGET_FRAMES)